<a href="https://colab.research.google.com/github/akimotolab/CMAES_Tutorial/blob/main/5_multimodality.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 多峰函数优化

前面没有特别强调的一点是：CMA-ES 中几乎所有算法超参数，例如协方差矩阵学习率、进化路径累积率、步长阻尼率等，都有根据问题维数 $N$ 和每代候选解数量 $\lambda$（种群规模）计算出的推荐值。种群规模本身也有常用推荐值，例如 $\lambda=3+\lfloor4\log(N)\rfloor$。因此，除明显依赖具体问题的初始分布参数（初始均值与初始步长）外，使用者通常不需要大量手工调参。对于单峰函数，初始分布的影响也相对有限，只要不极端不合理，一般都能有效搜索。

但在多峰目标函数上，以下因素会直接影响最终落入哪个局部最优：
* 种群规模；
* 初始分布参数。

而且一次搜索通常很难保证得到真正理想的解，因此**重启策略（restart strategy）**非常重要。本章学习多峰函数中种群规模、初始参数和重启策略的作用。

## 典型多峰测试函数

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

#### Rastrigin 函数

Rastrigin 函数定义为
$$f(x)=\sum_{i=1}^{N}x_i^2+10(1-\cos(2\pi x_i)).$$
它在各坐标接近整数的位置存在大量局部最优，常用定义域为 $[-5,5]^N$，全局最优解为 $x^*=0$。虽然微观上局部极值很多，但从宏观尺度看，二次项使整体地形呈向下收敛的大谷形状。这类问题常被称为“全局单峰”或具有“大谷结构（big valley structure）”。

In [ ]:
def rastrigin(x):
    a = 10
    return np.sum(x ** 2 + a * (1 - np.cos(2 * np.pi * x)))

In [ ]:
dx, dy = 0.1, 0.1
y, x = np.mgrid[slice(-5, 5 + dy, dy), slice(-5, 5 + dx, dx)]
z = np.zeros(x.shape)
for i in range(x.shape[0]):
    for j in range(x.shape[1]): z[i,j] = rastrigin(np.array([x[i,j], y[i,j]]))
plt.figure(figsize=(3,3)); plt.axes((0.1,0.1,0.85,0.85)); plt.axis('equal')
CS=plt.contour(x,y,z,10); plt.clabel(CS,inline=1,fontsize=10); plt.grid()

#### Bohachevsky 函数

Bohachevsky 也是典型的宏观全局单峰函数：
$$f(x)=\sum_{i=1}^{N-1}x_i^2+2x_{i+1}^2+0.3(1-\cos(3\pi x_i))+0.4(1-\cos(4\pi x_{i+1})).$$
常见定义域为 $[-15,15]^N$，最优解同样为 $x^*=0$。

In [ ]:
def bohachevsky(x):
    """Bohachevsky function, commonly evaluated on [-15, 15]."""
    uni = np.sum(x[:-1] ** 2 + 2 * x[1:] ** 2)
    multi = np.sum(0.3 * (1 - np.cos(3 * np.pi * x[:-1])) + 0.4 * (1 - np.cos(4 * np.pi * x[1:])))
    return uni + multi

In [ ]:
dx,dy=0.04,0.04
y,x=np.mgrid[slice(-2,2+dy,dy),slice(-2,2+dx,dx)]; z=np.zeros(x.shape)
for i in range(x.shape[0]):
    for j in range(x.shape[1]): z[i,j]=bohachevsky(np.array([x[i,j],y[i,j]]))
plt.figure(figsize=(3,3)); plt.axes((0.1,0.1,0.85,0.85)); plt.axis('equal'); plt.contour(x,y,z,50); plt.grid()

#### Skew Rastrigin 函数

利用 Rastrigin 函数可定义
$$f(x)=f_\text{rastrigin}(\tilde x),\qquad \tilde x=\begin{cases}x,&x\le0\\10x,&x>0.\end{cases}$$
定义域仍可取 $[-5,5]^N$。从宏观上看它仍有向全局最优方向下降的趋势，但在 $x^*=0$ 两侧的敏感度严重不对称：最优解附近一侧的函数值会迅速升高，因此搜索算法很容易把真正最优区域判断成“没有希望”的区域，这是一种典型的欺骗性结构（deceptive structure）。当最优解靠近约束边界时，不恰当的约束处理甚至会让普通 Rastrigin 在算法看来具有类似的偏斜结构。

这个问题非常困难。对于中等以上维数，想以高概率稳定找到全局最优通常并不现实。

In [ ]:
def skewrastrigin(x):
    y=x.copy(); y[y>0]*=10
    return rastrigin(y)

In [ ]:
dx,dy=0.06,0.06
y,x=np.mgrid[slice(-5,1+dy,dy),slice(-5,1+dx,dx)]; z=np.zeros(x.shape)
for i in range(x.shape[0]):
    for j in range(x.shape[1]): z[i,j]=skewrastrigin(np.array([x[i,j],y[i,j]]))
plt.figure(figsize=(3,3)); plt.axes((0.1,0.1,0.85,0.85)); plt.axis('equal'); plt.contour(x,y,z,30); plt.grid()

## Double-Sphere 函数
Double-Sphere 取两个 Sphere 谷的最小值：
$$f(x)=\min\left[\sum_{i=1}^{N}(x_i-a_i)^2,\;N+s\sum_{i=1}^{N}(x_i-b_i)^2\right].$$
常见定义域为 $[-10,10]^N$。它只有两个局部极小谷，但若设 $s<1$，真正全局最优谷的吸引区域会比另一个宽谷小得多。因此不同初始分布可能很容易落入宽而次优的谷。这类地形也被称为 UV 结构：一个入口宽的局部谷和一个入口窄的全局最优谷。

In [ ]:
def doublesphere(x):
    s=0.2; a=2.5; b=-np.sqrt((a**2-1)/s)
    f1=np.sum((x-a)**2); f2=len(x)+s*np.sum((x-b)**2)
    return np.fmin(f1,f2)

In [ ]:
dx,dy=0.5,0.5
y,x=np.mgrid[slice(-10,10+dy,dy),slice(-10,10+dx,dx)]; z=np.zeros(x.shape)
for i in range(x.shape[0]):
    for j in range(x.shape[1]): z[i,j]=doublesphere(np.array([x[i,j],y[i,j]]))
plt.figure(figsize=(3,3)); plt.axes((0.1,0.1,0.85,0.85)); plt.axis('equal'); plt.contour(x,y,z,50); plt.grid()

#### Schwefel 函数

Schwefel 是典型的全局多峰函数：
$$f(x)=\sum_{i=1}^{N}418.9829-x_i\sin(\sqrt{|x_i|}).$$
定义域为 $[-500,500]^N$，全局最优解为 $x^*=(420.9687,\dots,420.9687)$，最优函数值约为 0。与 Rastrigin 不同，Schwefel 在原点附近起伏较小，离原点越远起伏幅度越大，因此宏观结构并不会简单指向全局最优。

In [ ]:
def schwefel(x):
    a=418.9829
    pen=np.sum(np.fmax(np.abs(x)-500,0)**2)
    if pen>0:
        fx=1000+pen  # 为方便实验，在定义域之外给予较大惩罚值
    else:
        fx=np.sum(a-x*np.sin(np.sqrt(np.abs(x))))
    return fx

In [ ]:
dx,dy=5,5
y,x=np.mgrid[slice(-500,500+dy,dy),slice(-500,500+dx,dx)]; z=np.zeros(x.shape)
for i in range(x.shape[0]):
    for j in range(x.shape[1]): z[i,j]=schwefel(np.array([x[i,j],y[i,j]]))
plt.figure(figsize=(3,3)); plt.axes((0.1,0.1,0.85,0.85)); plt.axis('equal'); plt.contour(x,y,z,50); plt.grid()

## 数值实验

In [ ]:
class CMAES(object):
    """带 CSA 的 CMA Evolution Strategy。"""
    def __init__(self,func,init_mean,init_sigma,nsample):
        self.func=func; self.mean=init_mean; self.sigma=init_sigma
        self.N=self.mean.shape[0]                     # 搜索空间维数
        self.arx=np.zeros((nsample,self.N))*np.nan    # 候选解
        self.arf=np.zeros(nsample)*np.nan             # 候选解目标值
        self.D=np.ones(self.N) # 协方差特征值
        self.B=np.eye(self.N)  # 协方差特征向量
        self.C=np.dot(self.B*self.D,self.B.T)
        self.weights=np.zeros(nsample); self.weights[:nsample//4]=1.0/(nsample//4)
        self.ps=np.zeros(self.N); self.mueff=1.0/np.sum(self.weights**2)
        self.cs=(2+self.mueff)/(3+self.mueff+self.N)
        self.ds=1.0+self.cs+max(1.0,np.sqrt(self.mueff/self.N))
        self.chiN=np.sqrt(self.N)*(1.0-1.0/(4.0*self.N)+1.0/(21.0*self.N*self.N))
        self.cmu=self.mueff/(self.N**2/2+self.N+self.mueff)
        self.t=0; self.neval=0
    def sample(self):
        """生成候选解。"""
        self.arz=np.random.normal(size=self.arx.shape)
        self.ary=np.dot(np.dot(self.arz,self.B)*np.sqrt(self.D),self.B.T)
        self.arx=self.mean+self.sigma*self.ary
    def evaluate(self):
        """评估候选解。"""
        for i in range(self.arf.shape[0]): self.arf[i]=self.func(self.arx[i]); self.neval+=1
    def update_param(self):
        """更新参数。"""
        self.t+=1; idx=np.argsort(self.arf)
        self.ps=(1-self.cs)*self.ps+np.sqrt(self.cs*(2-self.cs)*self.mueff)*np.dot(self.weights,self.arz[idx])
        self.C=(1-self.cmu)*self.C+self.cmu*np.dot(self.ary[idx].T*self.weights,self.ary[idx])
        self.D,self.B=np.linalg.eigh(self.C)
        self.sigma=self.sigma*np.exp(self.cs/self.ds*(np.linalg.norm(self.ps)/self.chiN-1))
        self.mean+=np.dot(self.weights,self.arx[idx]-self.mean)
    @property
    def coordinate_std(self): return np.sqrt(np.diag(self.C))*self.sigma
    @property
    def lam(self): return len(self.arf)
    @property
    def xmean(self): return self.mean

In [ ]:
es=CMAES(func=rastrigin,init_mean=-5.0+10.0*np.random.rand(10),init_sigma=0.1,nsample=1000)
maxiter=100
mean=np.zeros(maxiter)*np.nan; sigmaN=np.zeros(maxiter)*np.nan
D=np.zeros((maxiter,es.N))*np.nan; diagC=np.zeros((maxiter,es.N))*np.nan
for i in range(maxiter):
    es.sample(); es.evaluate(); es.update_param()
    mean[i]=es.func(es.mean); sigmaN[i]=es.sigma*es.N; D[i]=es.D; diagC[i]=np.diag(es.C)
    if mean[i]<1e-8: break
plt.figure(figsize=(9,3)); plt.subplot(131); plt.semilogy(mean); plt.semilogy(sigmaN); plt.grid()
plt.subplot(132); plt.semilogy(D); plt.grid(); plt.subplot(133); plt.semilogy(diagC); plt.grid(); plt.tight_layout()

## 思考

对上面五个多峰函数分别改变以下参数，观察 CMA-ES 最终得到的局部最优与搜索过程，并解释原因。括号中的数值只是实验示例，不是必须采用的配置。

* 初始均值 `init_mean`：例如在各问题常用定义域中均匀随机初始化。
* 初始步长 `init_sigma`：例如定义域直径的 $1/4$、$1/40$、$1/400$。
* 种群规模 `nsample`：推荐值、$N$、$10N$、$100N$ 等。

其中 $N$ 是问题维数，修改 `init_mean` 的维数即可改变测试问题的维度。

## 重启策略

如前面的实验所示，在多峰问题中，不同种群规模与初始步长可能导致搜索收敛到完全不同的局部最优，而这些合适的数值无法事先知道。因此实践中常采用不断改变种群规模和初始分布的重启策略。

这里介绍 IPOP（Increasing POPulation）重启：初始均值在允许的初始化区域内随机生成，初始步长可设为该区域直径的约 $1/4$；种群规模从 CMA-ES 推荐值开始，每次重启翻倍。也可以随机改变初始步长，但这里为了简化保持固定。

实现重启时，单次运行的终止条件非常重要。下面给出一组考虑数值精度、停滞、协方差条件数与搜索尺度等因素的通用终止条件示例；实际工程中应根据具体问题进一步调整。

#### 终止条件实现

In [ ]:
from collections import deque
import math
class Checker:
    """BBOB termination checker for CMA-ES."""
    def __init__(self,cma):
        assert isinstance(cma,CMAES); self._cma=cma; self._init_std=cma.coordinate_std
        self._N=cma.N; self._lam=cma.lam
        self._hist_fbest=deque(maxlen=10+int(np.ceil(30*self._N/self._lam)))
        self._hist_feq_flag=deque(maxlen=self._N); self._hist_fmin=deque(); self._hist_fmed=deque()
    def __call__(self): return self.bbob_check()
    def check_maxiter(self): return self._cma.t>100+50*(self._N+3)**2/np.sqrt(self._lam)
    def check_tolhistfun(self):
        self._hist_fbest.append(np.min(self._cma.arf))
        return self._cma.t>=10+int(np.ceil(30*self._N/self._lam)) and np.max(self._hist_fbest)-np.min(self._hist_fbest)<1e-12
    def check_equalfunvals(self):
        k=int(math.ceil(0.1+self._lam/4)); sarf=np.sort(self._cma.arf); self._hist_feq_flag.append(sarf[0]==sarf[k]); return 3*sum(self._hist_feq_flag)>self._N
    def check_tolx(self): return np.all(self._cma.coordinate_std/self._init_std)<1e-12
    def check_tolupsigma(self): return np.any(self._cma.coordinate_std/self._init_std>1e3)
    def check_stagnation(self):
        self._hist_fmin.append(np.min(self._cma.arf)); self._hist_fmed.append(np.median(self._cma.arf))
        _len=int(np.ceil(self._cma.t/5+120+30*self._N/self._lam))
        if len(self._hist_fmin)>_len: self._hist_fmin.popleft(); self._hist_fmed.popleft()
        return self._cma.t>=_len and np.median(np.asarray(self._hist_fmin)[-20:])>=np.median(np.asarray(self._hist_fmed)[:20])
    def check_conditioncov(self): return np.max(self._cma.D)/np.min(self._cma.D)>1e14
    def check_noeffectaxis(self):
        t=self._cma.t%self._N; test=0.1*self._cma.sigma*np.sqrt(self._cma.D[t])*self._cma.B[:,t]
        return np.all(self._cma.xmean==self._cma.xmean+test)
    def check_noeffectcoor(self): return np.all(self._cma.xmean==self._cma.xmean+0.2*self._cma.coordinate_std)
    def check_flat(self): return np.max(self._cma.arf)==np.min(self._cma.arf)
    def bbob_check(self):
        checks=[(self.check_maxiter,'bbob_maxiter'),(self.check_tolhistfun,'bbob_tolhistfun'),(self.check_equalfunvals,'bbob_equalfunvals'),(self.check_tolx,'bbob_tolx'),(self.check_tolupsigma,'bbob_tolupsigma'),(self.check_stagnation,'bbob_stagnation'),(self.check_conditioncov,'bbob_conditioncov'),(self.check_noeffectaxis,'bbob_noeffectaxis'),(self.check_noeffectcoor,'bbob_noeffectcoor'),(self.check_flat,'bbob_flat')]
        for fn,name in checks:
            if fn(): return True,name
        return False,''

#### 执行代码

In [ ]:
N=10
lam=int(4+3*np.log(N))
init_mean=-5.0+(5.0-(-5.0))*np.random.randn(N)
init_sigma=(5.0-(-5.0))/4.0
NUM_RESTART=10; MAX_NEVAL=1e6; F_TARGET=1e-8; total_neval=0
maxiter=10000
mean=np.zeros(maxiter)*np.nan; sigmaN=np.zeros(maxiter)*np.nan
es=CMAES(func=rastrigin,init_mean=init_mean,init_sigma=init_sigma,nsample=lam); checker=Checker(es)
t=0
for restart in range(NUM_RESTART):
    issatisfied=False; fbestsofar=np.inf
    while not issatisfied:
        es.sample(); es.evaluate(); es.update_param()
        mean[t]=es.func(es.mean); sigmaN[t]=es.sigma*es.N; t+=1
        fbest=np.min(es.arf); fbestsofar=min(fbest,fbestsofar)
        if fbest<=F_TARGET: issatisfied,condition=True,'ftarget'
        elif t>=maxiter: issatisfied,condition=True,'maxiter'
        else: issatisfied,condition=checker()
        if es.t%10==0: print(es.t,es.neval,fbest,fbestsofar)
    print('Terminated with condition: '+str(condition))
    total_neval+=es.neval
    if total_neval<MAX_NEVAL and fbest>F_TARGET and restart+1<NUM_RESTART:
        popsize=lam*2**(restart+1)
        init_mean=-5.0+(5.0-(-5.0))*np.random.randn(N); init_sigma=(5.0-(-5.0))/4.0
        es=CMAES(func=rastrigin,init_mean=init_mean,init_sigma=init_sigma,nsample=popsize); checker=Checker(es)
        print('Restart with popsize: '+str(popsize))
    else: break
plt.semilogy(mean,label='f(mean)'); plt.semilogy(sigmaN,label='sigma*N'); plt.grid(); plt.legend()